# 第 1 周末练习 —— 技术问答解释器（DeepSeek + Ollama）

## 练习目标（理念）

为了展示你对 **OpenAI 兼容 API**（这里接 DeepSeek）以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一段代码或技术问题
- **输出**：清晰、简洁的解释（可配类比）
- **额外要求**：用**流式（streaming）**一边生成一边用 Markdown 刷新显示

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(..., stream=True)` |
| `messages`（system / user） | system 定讲解风格，user 放具体问题 |
| 流式输出 | `update_display` 逐步刷新 Markdown |
| 多后端 | DeepSeek 云端 + Ollama 本地（OpenAI 兼容 `/v1`） |

## 怎么跑

1. 准备 `.env`：至少有 `DEEPSEEK_API_KEY`；本地需 Ollama 并拉取 `llama3.2`
2. 从上到下运行；可在 user_prompt 格改写问题
3. 分别跑 DeepSeek 与 Llama 两格，对比回答风格


In [1]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables）
import os
# 导入标准库 json：本练习主流程未直接用到，保留原导入
import json
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、display / update_display 做流式刷新
from IPython.display import Markdown, display, update_display
# 从同目录 scraper 导入网页抓取函数（本 notebook 问答主路径未调用，保留原导入）
from scraper import fetch_website_links, fetch_website_contents
# 从 openai 导入 OpenAI 客户端：可指向 DeepSeek 或 Ollama 的 OpenAI 兼容端点
from openai import OpenAI


In [2]:
# ========== 两个后端的 base_url 常量（OpenAI 兼容 /v1）==========

# DeepSeek 云端 OpenAI 兼容接口根地址
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
# 本地 Ollama 的 OpenAI 兼容接口（默认 11434）
OLLAMA_BASE_URL = "http://localhost:11434/v1"


In [3]:
# ========== 模型名字常量：后面创建请求时只引用这些名字 ==========

# OpenAI 云端小模型名（本文件后半主路径实际用的是 DeepSeek；常量保留）
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2
MODEL_LLAMA = 'llama3.2'
# DeepSeek 对话模型 id
MODEL_DEEPSEEK = 'deepseek-chat'


In [4]:
# ========== 环境 + DeepSeek 客户端：读密钥、粗检格式、指向 DeepSeek base_url ==========

# 加载 .env；override=True 允许覆盖已有环境变量
load_dotenv(override=True)
# 读取 DeepSeek API Key（变量名必须是 DEEPSEEK_API_KEY）
api_key = os.getenv('DEEPSEEK_API_KEY')

# 粗检密钥形态：以 sk-proj- 开头且长度>10 时打印“看起来正常”（文案保持原样，便于对照排查笔记）
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# 再存一份模型名字符串（与 MODEL_DEEPSEEK 同为 deepseek-chat）
MODEL = 'deepseek-chat'
# 创建指向 DeepSeek 的 OpenAI 兼容客户端（变量名仍叫 openai，沿用课程习惯）
openai = OpenAI(base_url="https://api.deepseek.com/v1", api_key=api_key)


There might be a problem with your API key? Please visit the troubleshooting notebook!


In [14]:
# ========== 环境 + Ollama 客户端：同一套密钥读取逻辑，base_url 改指向本地 ==========

# 再次加载 .env（与上一格独立，方便单独重跑）
load_dotenv(override=True)
# 仍读 DEEPSEEK_API_KEY：给 OpenAI 兼容客户端一个 api_key 字段（Ollama 通常不校验，但 SDK 需要非空）
api_key = os.getenv('DEEPSEEK_API_KEY')

# 同样的粗检与提示文案（保持原样）
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# 本地路径也写一份 MODEL 名（后面 explain_llama 实际用 MODEL_LLAMA）
MODEL = 'deepseek-chat'
# 创建指向本地 Ollama /v1 的客户端，变量名 llama
llama = OpenAI(base_url=OLLAMA_BASE_URL, api_key=api_key)


There might be a problem with your API key? Please visit the troubleshooting notebook!


In [5]:
# ========== system prompt：规定「怎么解释」——英文指令保留，改译会改变回答风格 ==========

# 系统提示：专家编码助手 + 简洁解释 + 类比
system_prompt ="""You are an expert coding assistant and LLM engineer. 
Whatever I ask, explain clearly and concisely. 
Use simple examples or analogies to make your answers easy to understand."""

# 打印出来，方便确认当前生效的 system 文本
print(system_prompt)


You are an expert coding assistant and LLM engineer. 
Whatever I ask, explain clearly and concisely. 
Use simple examples or analogies to make your answers easy to understand.


In [6]:
# ========== user prompt：真正要问的问题（改这里即可换题）==========

# 这是问题；输入此内容以询问新问题
# 示例：解释一段含 yield from 与集合推导的 Python 代码
user_prompt = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [17]:
# ========== DeepSeek 流式讲解：边收 delta 边刷新 Markdown 显示 ==========

# 函数：用上面创建的 openai（DeepSeek）客户端做流式 Chat Completions
def explain(system_prompt, user_prompt):
    # stream=True：服务端持续推送增量，不要等整段完成
    stream = openai.chat.completions.create(
        model=MODEL_DEEPSEEK,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
          ],
        stream=True
    )    
    # 累积已收到的文本，用于每次整体刷新显示
    response = ""
    # 先占位一个空 Markdown，拿到 display_id 以便后续 update_display
    display_handle = display(Markdown(""), display_id=True)
    # 遍历流式 chunk：取出 delta.content（可能为 None），拼到 response 上并刷新
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [18]:
# ========== Ollama / Llama 流式讲解：接口形状与上一格相同，只换 client 与 model ==========

# 函数：用 llama 客户端 + MODEL_LLAMA 做同样的流式解释
def explain_llama(system_prompt, user_prompt):
    # 本地模型同样走 OpenAI 兼容的 chat.completions + stream
    stream = llama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
          ],
        stream=True
    )    
    # 同样累积文本并用 display_id 刷新
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# ========== 运行 DeepSeek：对流式 explain ==========

# 深度搜寻 / DEEPSEEK：把 system + user 交给云端 deepseek-chat
explain(system_prompt, user_prompt)


This code does two things:

**What it does:**
It yields each unique author name from a list of book dictionaries, but only if the book has an "author" field.

**Breaking it down:**

1. **The comprehension part** `{book.get("author") for book in books if book.get("author")}`
   - Creates a **set** (notice the curly braces `{}`)
   - Iterates over `books` collection
   - For each book, gets the author using `.get("author")` (returns `None` instead of error if missing)
   - The `if` condition filters out books without an author
   - Being a set, it automatically removes duplicate authors

2. **The `yield from`** 
   - Makes this function a generator
   - Delegates to the set's iterator
   - Yields each author one at a time, on demand

**Why use this pattern?**

```python
# Instead of:
def get_authors(books):
    authors = set()
    for book in books:
        if book.get("author"):
            authors.add(book.get("author"))
    return authors  # Returns all at once, wasting memory

# Use this:
def get_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}
    # Lazily yields authors, less memory

# Usage:
for author in get_authors(books):
    print(author)
```

**Key benefits:**
- **Memory efficient**: Doesn't need to store all books before processing
- **Duplicate free**: Set automatically removes duplicates
- **Safe**: `.get()` prevents KeyError if "author" is missing
- **Concise**: Single line does iteration, filtering, deduplication, and yielding

The main "why" is **lazy evaluation** - you process and yield results only as needed, which is especially useful with large datasets.

In [19]:
# ========== 运行 Llama：对同一问题做本地流式 explain ==========

# 骆驼 / LLAMA：对比本地 llama3.2 的回答风格与速度
explain_llama(system_prompt, user_prompt)


Let's break down the code step by step:

**What is it doing?**

This code is using a concept called "generator yields" to extract data from an iterable.

Here's what happens:

1. `yield from`: This keyword is used to delegate the responsibility of yielding values to another iterator (or generator) inside this function.
2. `{book.get("author") for book in books if book.get("author")}`: This is a dictionary comprehension that iterates over an iterable (`books`), filters it based on a condition, and extracts values.

**How does it work?**

Imagine you have a list of bookstore objects (`books`) with each object having an `author` attribute. You want to extract the authors from all the books in this list.

The dictionary comprehension works as follows:

1. Iterate over each book object in `books`.
2. For each book, check if it has an `author` attribute using `book.get("author")`. This is a safety check to ensure that only valid objects with the `author` attribute are processed.
3. If the book has an author, extract the author's name from the book object (e.g., `"John Doe"`).

The generator uses `yield from` to delegate this processing to another iterator (which would be created by the dictionary comprehension itself). This allows us to avoid storing all the data in memory at once.

**Outcome**

The `yield from ...` expression will yield each author's name, one by one. The receiver of this code (typically a coroutine or an asynchronous function) can then choose whether to iterate over and use these values, without having to store them all initially.

Here's a simple example to illustrate the difference between storing data in memory vs. yielding values:

**Before `yield from`:**
```python
books = [Book1("John Doe"), Book2("Jane Smith")]
authors = []
for book in books:
    authors.extend(book.author)
```
Stores all authors in memory (`authors` list).

**After `yield from`:**
```python
def get_authors():
    for book in books if book.get("author"):
        yield book.get("author")

authors = []
for author in get_authors():
    authors.append(author)
```
Yields each author's name one by one, without storing them all initially.

I hope this explanation helps!